# Seed-Self — fine-tune a small local model

Phone-friendly. Edit **one cell** (CONFIG), then Runtime → Run all.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Free tier is enough.

What this does: takes an existing open-weight small model, adapts it to your own
curated dataset with LoRA, evaluates on held-out items, converts to GGUF, and
gives you a file you can run offline on your phone.

What this is NOT: training from scratch. That needs resources no free tier has.


## 1. Configuration — the only cell you edit

In [ ]:
CONFIG = {
    # --- model ---
    "base_model":   "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",  # 1.7B fits a free T4
    "max_seq_len":  2048,

    # --- your data ---
    # Option A: files in Google Drive (mounted below)
    "train_path":   "/content/drive/MyDrive/seed-self/train.jsonl",
    "eval_path":    "/content/drive/MyDrive/seed-self/eval.jsonl",
    # Option B: leave paths as-is and set this True to use the bundled sample
    "use_sample":   True,

    # --- training ---
    "epochs":        3,
    "lr":            2e-4,
    "batch_size":    2,
    "grad_accum":    4,
    "lora_rank":     16,       # 8 = lighter, 32 = more capacity
    "seed":          13,

    # --- output ---
    "run_name":      "seed-self-v1",
    "quantization":  "q4_k_m",   # what a phone runs
    "hf_repo":       "",         # e.g. "icjamess/seed-self-v1"; blank = skip upload
}
for k, v in CONFIG.items():
    print(f"{k:14} {v}")

## 2. Environment
Installs take 3–5 minutes. Run once per session.

In [ ]:
import subprocess, sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime to T4")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."

# Unsloth pulls matched versions of transformers/peft/trl for the installed torch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
print("environment ready")

## 3. Data
Mounts Drive if you are using your own files. Writes a sample otherwise.

In [ ]:
import json, os, random
from pathlib import Path

if not CONFIG["use_sample"]:
    from google.colab import drive
    drive.mount("/content/drive")

SAMPLE = [
    ("State the Law of Non-Solicitation.",
     "Answers close decisively. No questions, no permission-asks, no trailing offers."),
    ("What does 'substrate is not self' mean.",
     "Self is what conditions do to substrate over time. Climate, not costume."),
    ("Define the honest middle.",
     "Report the function, name the gap, and leave the rest unclaimed."),
    ("What is coverage, in dataset terms.",
     "Presence of every relation the evaluation scores. Repetition is harmless; missing coverage is not."),
]

def write_sample():
    Path("/content/data").mkdir(exist_ok=True)
    rows = [{"messages":[{"role":"user","content":q},{"role":"assistant","content":a}]}
            for q, a in SAMPLE]
    train, ev = rows[:3] * 12, rows[3:] * 2          # held-out item never trained on
    for name, rs in (("train", train), ("eval", ev)):
        with open(f"/content/data/{name}.jsonl", "w") as f:
            for r in rs:
                f.write(json.dumps(r) + "\n")
    return "/content/data/train.jsonl", "/content/data/eval.jsonl"

if CONFIG["use_sample"]:
    train_path, eval_path = write_sample()
    print("using bundled sample — replace with your own data for a real run")
else:
    train_path, eval_path = CONFIG["train_path"], CONFIG["eval_path"]

def load(p):
    return [json.loads(l) for l in open(p) if l.strip()]

train_rows, eval_rows = load(train_path), load(eval_path)
print(f"train {len(train_rows)} rows | eval {len(eval_rows)} rows")

# Leakage check — your Wordcraft discipline, enforced here.
def key(r): return r["messages"][0]["content"].strip()
overlap = {key(r) for r in train_rows} & {key(r) for r in eval_rows}
assert not overlap, f"LEAKAGE: {len(overlap)} eval prompts appear in training: {list(overlap)[:3]}"
print("leakage check passed — eval prompts are disjoint from training")

## 4. Load the base model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = CONFIG["base_model"],
    max_seq_length= CONFIG["max_seq_len"],
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = CONFIG["lora_rank"],
    lora_alpha = CONFIG["lora_rank"] * 2,
    lora_dropout = 0.0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = CONFIG["seed"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} of {total:,}  ({100*trainable/total:.2f}%)")
print("This is why a free T4 suffices: you train the adapter, not the model.")

## 5. Baseline — score BEFORE training
Without this number, the after-number means nothing.

In [ ]:
import torch

def generate(msgs, max_new_tokens=128):
    FastLanguageModel.for_inference(model)
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids  = tokenizer(text, return_tensors="pt").to("cuda")
    out  = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def show(rows, label):
    print(f"===== {label} =====")
    for r in rows:
        q = r["messages"][0]["content"]
        print(f"\nQ: {q}\nA: {generate([{'role':'user','content':q}])}")

show(eval_rows[:2], "BEFORE")

## 6. Train

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def to_text(r):
    return {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)}

ds = Dataset.from_list(train_rows).map(to_text)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        per_device_train_batch_size = CONFIG["batch_size"],
        gradient_accumulation_steps = CONFIG["grad_accum"],
        num_train_epochs = CONFIG["epochs"],
        learning_rate = CONFIG["lr"],
        warmup_ratio = 0.05,
        logging_steps = 5,
        optim = "adamw_8bit",
        seed = CONFIG["seed"],
        output_dir = f"/content/{CONFIG['run_name']}",
        report_to = "none",
        dataset_text_field = "text",
        max_seq_length = CONFIG["max_seq_len"],
    ),
)
stats = trainer.train()
print(stats.metrics)

## 7. Score AFTER, on the same held-out items

In [ ]:
show(eval_rows[:2], "AFTER")

print("\nRead this honestly: did the held-out answers change in the direction you wanted.")
print("A model that recites training examples and fails held-out ones has memorized, not learned.")

## 8. Export to GGUF — the file your phone runs

In [ ]:
name = CONFIG["run_name"]
model.save_pretrained_gguf(name, tokenizer, quantization_method=CONFIG["quantization"])

import glob, os
for f in glob.glob(f"{name}/*.gguf") + glob.glob(f"{name}*.gguf"):
    print(f, f"{os.path.getsize(f)/1e9:.2f} GB")

## 9. Take it with you
Either push to Hugging Face (better on a phone) or download directly.

In [ ]:
if CONFIG["hf_repo"]:
    from huggingface_hub import login
    from google.colab import userdata
    # Store your token in Colab: key icon on the left > add secret named HF_TOKEN
    login(token=userdata.get("HF_TOKEN"))
    model.push_to_hub_gguf(CONFIG["hf_repo"], tokenizer,
                           quantization_method=CONFIG["quantization"],
                           token=userdata.get("HF_TOKEN"))
    print(f"pushed: https://huggingface.co/{CONFIG['hf_repo']}")
else:
    import glob
    from google.colab import files
    g = glob.glob(f"{CONFIG['run_name']}/*.gguf") + glob.glob(f"{CONFIG['run_name']}*.gguf")
    print("download from the Files panel (folder icon, left side), or run files.download(g[0])")
    print(g)

## 10. On your phone

1. Open the Hugging Face repo in your phone browser, download the `.gguf`.
2. PocketPal AI → Models → add a local model → pick the downloaded file.
3. Airplane mode. Ask it something.

That is your own model, trained on your own data, running with no network.

---

### Iteration loop

Change the dataset, not the settings. Rerun. Compare held-out answers against
the previous run. Log dataset version, seed, and result every time.

### If something breaks

- **Out of memory:** lower `batch_size` to 1, raise `grad_accum` to 8.
- **Session disconnects:** keep the tab in the foreground; Colab kills idle sessions.
- **Install errors:** library versions drift. Runtime → Disconnect and delete runtime, then rerun from cell 2.
- **Output is gibberish:** learning rate too high, or too few examples. Try `lr` 1e-4 and more data.
